# 美崙之星 Hedonic Pricing Model 回歸

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

PROJECT_DIR = Path("/Users/xinc/GitHub/Personal-Project/ndhu/房地產")
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from utils import (
    NUMERIC_COLUMNS,
    describe_columns,
    ols,
    predict_price,
    prepare_model_rows,
    print_summary,
    read_xlsx_records,
)

DATA_PATH = PROJECT_DIR / "591_實價登錄整理.xlsx"
print(DATA_PATH.exists(), DATA_PATH)


True /Users/xinc/GitHub/Personal-Project/ndhu/房地產/591_實價登錄整理.xlsx


## 讀取 Excel

In [ ]:
records = read_xlsx_records(DATA_PATH)
print(f"Loaded {len(records)} rows")
print("Columns:", list(records[0]))
for row in records[:3]:
    print(row)

Loaded 46 rows
Columns: ['Case_ID', '成交年月', '成交日期', '成交總價_含車位_萬元', '建坪_含車位', '樓層', '總樓層', '屋齡', '屋齡平方', '有車位虛擬變數', '中高樓層虛擬變數', '地址', '到東華附小距離(google)', '到球崙公園距離(google)', '到門諾距離(google)']
{'Case_ID': '7256601', '成交年月': '115-02', '成交日期': '2026-02-01', '成交總價_含車位_萬元': '1880', '建坪_含車位': '67.400000000000006', '樓層': '5', '總樓層': '9', '屋齡': '9.31', '屋齡平方': '86.676100000000005', '有車位虛擬變數': '1', '中高樓層虛擬變數': '1', '地址': '中美路111號 | 5樓之1', '到東華附小距離(google)': '700', '到球崙公園距離(google)': '1300', '到門諾距離(google)': '750'}
{'Case_ID': '7121628', '成交年月': '114-10', '成交日期': '2025-10-13', '成交總價_含車位_萬元': '670', '建坪_含車位': '25.5', '樓層': '1', '總樓層': '9', '屋齡': '9', '屋齡平方': '81', '有車位虛擬變數': '1', '中高樓層虛擬變數': '0', '地址': '中美路111號之6 | 1,2樓', '到東華附小距離(google)': '700', '到球崙公園距離(google)': '1300', '到門諾距離(google)': '750'}
{'Case_ID': '7091875', '成交年月': '114-06', '成交日期': '2025-06-08', '成交總價_含車位_萬元': '1870', '建坪_含車位': '65.400000000000006', '樓層': '4', '總樓層': '9', '屋齡': '8.66', '屋齡平方': '74.995599999999996', '有車位虛擬變數': '1', '中高樓層

## 變數清理與模型設定

同一棟建物的 `總樓層` 與三個距離欄位沒有變異，無法在回歸中估計效果。`成交年分數` 與 `屋齡` 在單一建物樣本中高度重疊，因此主模型保留屋齡與屋齡平方；另外提供一個「時間趨勢、無屋齡」的敏感度模型作交叉檢查。

In [ ]:
model_rows = prepare_model_rows(records)

for item in describe_columns(model_rows, NUMERIC_COLUMNS + ["成交年分數"]):
    print(
        f"{item['欄位']}: "
        f"min={item['min']:,.4g}, "
        f"max={item['max']:,.4g}, "
        f"unique={item['unique']}"
    )

成交總價_含車位_萬元: min=670, max=2,838, unique=33
建坪_含車位: min=25.5, max=102.2, unique=21
樓層: min=1, max=9, unique=8
總樓層: min=9, max=9, unique=1
屋齡: min=0.15, max=9.31, unique=23
屋齡平方: min=0.0225, max=86.68, unique=23
有車位虛擬變數: min=0, max=1, unique=2
中高樓層虛擬變數: min=0, max=1, unique=2
到東華附小距離(google): min=700, max=700, unique=1
到球崙公園距離(google): min=1,300, max=1,300, unique=1
到門諾距離(google): min=750, max=750, unique=1
成交年分數: min=2,017, max=2,026, unique=23


## 主模型：log-linear hedonic model

主模型：`ln(總價) = 建坪 + 樓層 + 屋齡 + 屋齡平方 + 車位 + 中高樓層`。


In [ ]:
main_features = [
    "建坪_含車位",
    "樓層",
    "屋齡",
    "屋齡平方",
    "有車位虛擬變數",
    "中高樓層虛擬變數",
]

main_result = ols(model_rows, "ln_price", main_features)
print_summary(main_result)

n = 46, R^2 = 0.9095, Adjusted R^2 = 0.8956, RMSE(log) = 0.0708
----------------------------------------------------------------------------------------------------------------
變數                                  係數           標準誤          t值          p值         約略%影響
----------------------------------------------------------------------------------------------------------------
截距                            6.146262      0.095550      64.325      0.0000              
建坪_含車位                        0.017587      0.001000      17.582      0.0000         1.77%
樓層                            0.027021      0.007923       3.410      0.0006         2.74%
屋齡                            0.022517      0.016801       1.340      0.1802         2.28%
屋齡平方                         -0.000950      0.001995      -0.476      0.6340        -0.09%
有車位虛擬變數                      -0.052475      0.036648      -1.432      0.1522        -5.11%
中高樓層虛擬變數                     -0.093696      0.044529      -2.104      0.0

## 敏感度模型：時間趨勢、無屋齡

因為同一建物的屋齡會隨成交時間一起增加，這個版本改放 `成交年分數`，不放屋齡，用來觀察時間趨勢下的估價差異。


In [ ]:
time_features = [
    "成交年分數",
    "建坪_含車位",
    "樓層",
    "有車位虛擬變數",
    "中高樓層虛擬變數",
]

time_result = ols(model_rows, "ln_price", time_features)
print_summary(time_result)


n = 46, R^2 = 0.9089, Adjusted R^2 = 0.8976, RMSE(log) = 0.0710
----------------------------------------------------------------------------------------------------------------
變數                                  係數           標準誤          t值          p值         約略%影響
----------------------------------------------------------------------------------------------------------------
截距                          -23.675816      8.809478      -2.688      0.0072              
成交年分數                         0.014792      0.004350       3.401      0.0007         1.49%
建坪_含車位                        0.017570      0.000990      17.746      0.0000         1.77%
樓層                            0.027658      0.007734       3.576      0.0003         2.80%
有車位虛擬變數                      -0.058073      0.034388      -1.689      0.0913        -5.64%
中高樓層虛擬變數                     -0.097423      0.043402      -2.245      0.0248        -9.28%


## 標的物估價示範

PDF 標的物為中美路 111 號、約 67.44 坪、中高樓層華廈。下方示範先假設 5 樓、9.3 年屋齡、有車位；實際樓層、屋齡、車位可以改 `target`。


In [ ]:
target = {
    "成交年分數": 2026 + (5 - 1) / 12,
    "建坪_含車位": 67.44,
    "樓層": 5,
    "屋齡": 9.3,
    "屋齡平方": 9.3 ** 2,
    "有車位虛擬變數": 1,
    "中高樓層虛擬變數": 1,
}

for label, result, features in [
    ("主模型", main_result, main_features),
    ("時間趨勢敏感度模型", time_result, time_features),
]:
    ln_pred, price_pred = predict_price(result, features, target)
    print(f"{label}: ln(總價)={ln_pred:.4f}, 總價={price_pred:,.0f} 萬元, 單價={price_pred / target['建坪_含車位']:,.2f} 萬元/坪")


主模型: ln(總價)=7.4485, 總價=1,717 萬元, 單價=25.46 萬元/坪
時間趨勢敏感度模型: ln(總價)=7.4648, 總價=1,746 萬元, 單價=25.88 萬元/坪


## 報告寫法提醒

- 這份資料有 46 筆實價登錄。主模型的距離變數與總樓層因為沒有變異，無法估計。
- 同一棟建物中，成交時間與屋齡高度重疊，不建議同時放在同一個模型裡硬解釋。
- 最終估價建議搭配 5 個 comps 的 Sales Comparison Adjustments 表交叉檢查。
